In [ ]:
dfp = spark.read.parquet(PARQUET_DIR)

from pyspark.sql.functions import broadcast
label_freq = dfp.groupBy("label").count().withColumnRenamed("count", "label_freq")
dfp = dfp.join(broadcast(label_freq), on="label", how="left")

from pyspark import StorageLevel
dfp = dfp.persist(StorageLevel.MEMORY_AND_DISK)
_ = dfp.count()

years = dfp.select("year").distinct().orderBy("year")
year_list = [int(r["year"]) for r in years.collect() if r["year"] is not None]
year_list.sort()

train_end = int(len(year_list) * 0.7)
valid_end = int(len(year_list) * 0.85)

train_years = year_list[:train_end]
valid_years = year_list[train_end:valid_end]
test_years  = year_list[valid_end:]

train_df = dfp.filter(F.col("year").isin(train_years))
valid_df = dfp.filter(F.col("year").isin(valid_years))
test_df  = dfp.filter(F.col("year").isin(test_years))

print("Train/Valid/Test:", train_df.count(), valid_df.count(), test_df.count())